# Stage 1b — Multi-Label Validation

**AAI-590 Capstone · Monish Yarapathineni**

Run top to bottom in a fresh Colab runtime. Self-contained.

---

### Background

The single-label pilot (notebook 03) reached **κ = 0.560**. Diagnosis: a single
wrong answer underdetermines which misconception produced it, so forcing one
label makes the model choose arbitrarily between defensible options. Full record
in `capstone/stage1-pilot-findings.md`.

This notebook tests whether allowing **multiple labels per answer** fixes that.

### Results already established (2026-07-31)

| Finding | Value |
|---|---|
| Multi-label Jaccard, 2 passes | 0.694 |
| Macro per-category κ | **0.661** (vs 0.560 single-label) |
| Mean label-set size | 1.58 |
| Per-category binary probe, mean true | 1.50 — confirms 1–2 is honest, not a format artifact |

Sections 6 and 7 reproduce these. **Section 8 (discrimination) is the open
question** and the reason to run this notebook.

### The three conditions

| # | Condition | Threshold | Status |
|---|---|---|---|
| 1 | Reliability beats single-label | macro κ > 0.560 | ✅ 0.661 |
| 2 | Label sets stay small | mean size 1.0–3.0 | ✅ 1.58 |
| 3 | Per-student profiles differ | overdispersion > 1.0 | ❓ **untested** |

Condition 3 decides whether any of this is useful. Reliable labels that give every
student the same profile would be worthless.

### Note on metrics

An earlier version gated condition 1 on raw Jaccard > 0.75 and reported FAIL at
0.718. That was wrong — Jaccard is not corrected for chance, the same error as
using raw agreement to choose taxonomy granularity in notebook 03. Macro
per-category Cohen's κ is the correct comparison against the κ = 0.560 baseline.

---

## 1 · Setup

In [ ]:
try:
    import google.colab
    IN_COLAB = True
    %pip install -q datasets huggingface_hub anthropic
except ImportError:
    IN_COLAB = False

import os, re, json, time, textwrap, random, itertools
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset, load_from_disk
from anthropic import Anthropic

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
plt.rcParams.update({'figure.dpi': 130, 'figure.figsize': (9, 4)})

RANDOM_SEED = 590          # same seed as notebook 03 — reproduces the same 200
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)

# ---- Tunables --------------------------------------------------------------
BATCH_SIZE   = 10          # small: extended thinking eats the output budget
MAX_TOKENS   = 24000
N_STUDENTS   = 40          # Section 8 — raise for a tighter estimate, costs more
MIN_WRONG    = 20          # skip students with fewer labelable wrong answers
RUN_PROBE    = False       # Section 7 — already answered; set True to redo

print(f'Colab: {IN_COLAB} | batch {BATCH_SIZE} | max_tokens {MAX_TOKENS}')

## 2 · Load taxonomy

In [ ]:
TAXONOMY_PATH = "misconception_taxonomy_v1.json"

if not os.path.exists(TAXONOMY_PATH) and IN_COLAB:
    from google.colab import files
    print("Upload misconception_taxonomy_v1.json (from notebook 02):")
    up = files.upload()
    TAXONOMY_PATH = list(up.keys())[0]

with open(TAXONOMY_PATH) as f:
    artifact = json.load(f)

taxonomy       = artifact["taxonomy"]
categories     = taxonomy["categories"]
category_names = [c["name"] for c in categories]

print(f"{len(category_names)} categories:")
for n in category_names:
    print("  •", n)

## 3 · Load FoundationalASSIST

In [ ]:
if IN_COLAB:
    from huggingface_hub import login
    login()

HF_REPO = 'ASSISTments/FoundationalASSIST'
LOCAL_BASE = os.path.expanduser('~/Desktop/Educator/aai590-capstone/data/foundationalassist')

if IN_COLAB:
    problems_ds     = load_dataset(HF_REPO, 'Foundational ASSIST Dataset')
    interactions_ds = load_dataset(HF_REPO, 'Interactions')
else:
    problems_ds     = load_from_disk(os.path.join(LOCAL_BASE, 'Foundational ASSIST Dataset'))
    interactions_ds = load_from_disk(os.path.join(LOCAL_BASE, 'interactions'))

problems     = problems_ds['train'].to_pandas()
interactions = interactions_ds['train'].to_pandas()
print(f'Problems: {problems.shape}   Interactions: {interactions.shape}')

## 4 · Rebuild misconception rows and labeling pairs

Same definition as notebooks 01 and 03: score of zero **and** submitted answer
differs from correct. Select-all excluded — a wrong subset can encode several
errors simultaneously, so it cannot carry a single coherent label.

The student column is `user_id`, not `student_id`.

In [ ]:
def strip_html(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r'xmlns="[^"]*"', '', text)
    text = re.sub(r'<mfrac>\s*<mn>([^<]+)</mn>\s*<mn>([^<]+)</mn>\s*</mfrac>', r'\1/\2', text)
    text = re.sub(r'<msup>\s*<mi>([^<]+)</mi>\s*<mn>([^<]+)</mn>\s*</msup>', r'\1^\2', text)
    text = re.sub(r'<msup>\s*<mn>([^<]+)</mn>\s*<mn>([^<]+)</mn>\s*</msup>', r'\1^\2', text)
    text = re.sub(r'<mo>([^<]+)</mo>', r' \1 ', text)
    text = re.sub(r'<m[a-z]+>([^<]*)</m[a-z]+>', r'\1', text)
    text = re.sub(r'<[^>]+>', ' ', text)
    for ent, ch in {'&nbsp;': ' ', '&lt;': '<', '&gt;': '>', '&amp;': '&',
                    '&le;': '≤', '&ge;': '≥', '&deg;': '°', '&times;': '×'}.items():
        text = text.replace(ent, ch)
    text = re.sub(r'&#\d+;', '', text)
    text = re.sub(r'&[a-z]+;', '', text)
    return re.sub(r'\s+', ' ', text).strip()


def normalise_answer(s):
    return "" if pd.isna(s) else str(s).strip().lower()


def norm_answer_key(s):
    """Collapse delimiter whitespace so identical answers group together."""
    if pd.isna(s):
        return ""
    s = re.sub(r'\s*,\s*', ' , ', str(s).strip())
    return re.sub(r'\s+', ' ', s)


merged = interactions.merge(
    problems[['problem_id', 'Problem Type', 'Answer Types', 'Fill-in Answers',
              'Multiple Choice Answers', 'Multiple Choice Options', 'Problem Body']],
    on='problem_id', how='left'
)

fillin_wrong = merged[
    (merged['Problem Type'] == 'Fill-in-the-blank(s)') &
    (merged['discrete_score'] == 0) &
    merged.apply(lambda r: normalise_answer(r['answer_text'])
                 != normalise_answer(r['Fill-in Answers']), axis=1)
].copy()

mc1_wrong = merged[
    (merged['Problem Type'] == 'Multiple Choice (select 1)') &
    (merged['discrete_score'] == 0) &
    merged.apply(lambda r: normalise_answer(r['answer_text'])
                 != normalise_answer(r['Multiple Choice Answers']), axis=1)
].copy()

fillin_wrong['answer_key'] = fillin_wrong['answer_text'].map(norm_answer_key)
mc1_wrong['answer_key']    = mc1_wrong['answer_text'].map(norm_answer_key)

print(f'Fill-in wrong rows : {len(fillin_wrong):,}')
print(f'MC-1 wrong rows    : {len(mc1_wrong):,}')

In [ ]:
TOP_N_FILLIN = 10

fillin_pairs = (fillin_wrong.groupby(['problem_id', 'answer_key'])
                .size().reset_index(name='n_students')
                .rename(columns={'answer_key': 'answer_text'}))
fillin_pairs['rank'] = (fillin_pairs.groupby('problem_id')['n_students']
                        .rank(method='first', ascending=False))
fillin_pairs = fillin_pairs[fillin_pairs['rank'] <= TOP_N_FILLIN]

mc_pairs = (mc1_wrong.groupby(['problem_id', 'answer_key'])
            .size().reset_index(name='n_students')
            .rename(columns={'answer_key': 'answer_text'}))

pmeta = problems.set_index('problem_id')

def build_triples(pairs, fmt):
    rows = []
    for _, r in pairs.iterrows():
        try:
            p = pmeta.loc[r['problem_id']]
        except KeyError:
            continue
        if isinstance(p, pd.DataFrame):
            p = p.iloc[0]
        correct = p['Fill-in Answers'] if fmt == 'fill-in' else p['Multiple Choice Answers']
        body = strip_html(p['Problem Body'])
        if not body or pd.isna(correct):
            continue
        rows.append({
            'problem_id': r['problem_id'], 'format': fmt,
            'problem_text': body, 'correct': strip_html(correct),
            'wrong': strip_html(r['answer_text']),
            'options': strip_html(p['Multiple Choice Options']) if fmt == 'mc' else '',
            'n_students': r['n_students'],
        })
    return pd.DataFrame(rows)

fillin_triples = build_triples(fillin_pairs, 'fill-in')
mc_triples     = build_triples(mc_pairs, 'mc')

print(f'Fill-in triples : {len(fillin_triples):,}')
print(f'MC triples      : {len(mc_triples):,}')
print(f'TOTAL           : {len(fillin_triples) + len(mc_triples):,}   (expect ~20,602)')

### 4.1 · Reproduce the notebook 03 pilot sample

In [ ]:
N_PER_FORMAT = 100

pilot = pd.concat([
    fillin_triples.sample(min(N_PER_FORMAT, len(fillin_triples)), random_state=RANDOM_SEED),
    mc_triples.sample(min(N_PER_FORMAT, len(mc_triples)), random_state=RANDOM_SEED),
]).reset_index(drop=True)

print(f'Pilot: {len(pilot)} items — {pilot["format"].value_counts().to_dict()}')
print('Same seed as notebook 03, so comparable to the kappa = 0.560 baseline.')

## 5 · Client and robust labeling machinery

Three failure modes seen in earlier runs, all handled here:

- **`temperature` is deprecated** on current models — not passed. Labeling is
  therefore not deterministic, which is why self-agreement is measured rather
  than assumed.
- **Responses begin with a thinking block**, so `resp.content[0].text` fails.
  `response_text()` scans for text blocks instead.
- **Extended thinking can consume the whole output budget**, leaving a response
  with *no* text block at all. Handled by small batches, a high `max_tokens`, an
  explicit `stop_reason` check, and splitting any failed batch in half rather
  than dropping it.

In [ ]:
try:
    from google.colab import userdata
    api_key = userdata.get("ANTHROPIC_API_KEY")
except ImportError:
    api_key = os.environ.get("ANTHROPIC_API_KEY")

assert api_key, "ANTHROPIC_API_KEY not found — add it to Colab secrets."
client = Anthropic(api_key=api_key)
LABELING_MODEL = "claude-sonnet-5"


def response_text(resp):
    """Concatenate text blocks, skipping thinking blocks."""
    parts = [b.text for b in resp.content if getattr(b, "type", None) == "text"]
    if not parts:
        raise ValueError(
            f"No text block (budget exhausted by thinking?); "
            f"blocks={[getattr(b,'type','?') for b in resp.content]}"
        )
    return "".join(parts)


def extract_json(text):
    cand = None
    fenced = re.search(r"```(?:json)?\s*(.*?)\s*```", text, re.S)
    if fenced:
        cand = fenced.group(1)
        try:
            return json.loads(cand)
        except json.JSONDecodeError:
            cand = None
    if cand is None:
        try:
            cand = text[text.index("{"):text.rindex("}") + 1]
        except ValueError:
            raise ValueError(f"No JSON found:\n{text[:600]}")
    try:
        return json.loads(cand)
    except json.JSONDecodeError:
        return json.loads(re.sub(r"}\s*{", "},{", cand))


def as_set(v):
    """Model output -> set of valid category names. Reports nothing silently."""
    if isinstance(v, str):
        v = [v]
    return set(x for x in (v or []) if x in category_names or x == 'UNASSIGNABLE')


def count_dropped(raw, n):
    """How many returned labels failed to match a category name exactly."""
    d = 0
    for i in range(n):
        for x in (raw.get(str(i + 1)) or []):
            if x not in category_names and x != 'UNASSIGNABLE':
                d += 1
    return d


print("Client ready:", LABELING_MODEL)

### 5.1 · The prompt

Deliberately free of any hint about how many labels to return. An earlier version
said *"most reveal one or two"* and showed a schema example with two entries then
one — the model reproduced that pattern exactly, producing a suspiciously precise
100/100 split. Section 7 confirmed via independent per-category questioning that
1–2 is the model's honest judgment, not a format artifact.

In [ ]:
MULTI_TEMPLATE = """You are labeling student wrong answers with the misconception \
type(s) they reveal.

CATEGORIES:
{cats}

For each numbered item you get the problem, the correct answer, and what the \
student submitted.

Rules:
- Select every category that genuinely applies to this specific wrong answer. \
Some answers reveal a single clear misconception; others are consistent with \
several at once. Judge each item on its own evidence.
- Include a category only if you would defend it against a colleague. Do not add \
categories to hedge, and do not omit a category that genuinely fits.
- There is no expected number of labels. Return as many or as few as the evidence \
supports.
- If none genuinely applies, return ["UNASSIGNABLE"].
- Order most to least likely.

Return ONLY a JSON object mapping each item number to an array of category names.
Example shows format only — the number of entries carries no meaning:
{{"1": [...], "2": [...], "3": [...]}}

ITEMS:
{items}"""

cats_block = "\n".join(f"- {c['name']}: {c['definition']}" for c in categories)


def format_item(i, r):
    lines = [f"{i}. PROBLEM: {textwrap.shorten(r['problem_text'], 400)}"]
    if r["options"]:
        lines.append(f"   OPTIONS: {textwrap.shorten(r['options'], 300)}")
    lines.append(f"   CORRECT ANSWER: {r['correct']}")
    lines.append(f"   STUDENT SUBMITTED: {r['wrong']}")
    return "\n".join(lines)


def label_batch(df_batch, start_idx, template=None, max_retries=2):
    """Streaming call. The SDK refuses non-streaming requests whose max_tokens
    implies a possible >10 min generation, so streaming is used unconditionally."""
    tmpl = template or MULTI_TEMPLATE
    items = "\n\n".join(format_item(start_idx + i + 1, r)
                         for i, (_, r) in enumerate(df_batch.iterrows()))
    last = None
    for attempt in range(max_retries):
        try:
            with client.messages.stream(
                model=LABELING_MODEL,
                max_tokens=MAX_TOKENS,
                messages=[{"role": "user", "content":
                           tmpl.format(cats=cats_block, items=items)}],
            ) as stream:
                final = stream.get_final_message()
            if final.stop_reason == "max_tokens":
                raise ValueError(f"hit max_tokens on batch of {len(df_batch)}")
            return extract_json(response_text(final))
        except Exception as e:
            last = e
            time.sleep(2 ** attempt)
    raise last


def run_pass(df, tag="", template=None, batch_size=None):
    """Label a dataframe. A failing batch is split in half and retried."""
    bs = batch_size or BATCH_SIZE
    out, failed = {}, []
    for start in range(0, len(df), bs):
        chunk = df.iloc[start:start + bs]
        try:
            out.update(label_batch(chunk, start, template))
        except Exception as e:
            if len(chunk) > 1:
                mid = len(chunk) // 2
                for off, sub in ((0, chunk.iloc[:mid]), (mid, chunk.iloc[mid:])):
                    try:
                        out.update(label_batch(sub, start + off, template))
                    except Exception as e2:
                        failed.append(start + off)
                        print(f"\n  {tag} sub-batch {start+off} failed: {e2}")
            else:
                failed.append(start)
                print(f"\n  {tag} item {start} failed: {e}")
        print(f"\r  {tag} {len(out):,} / {len(df):,}", end="")
    print()
    if failed:
        print(f"  {tag} unresolved: {len(failed)} batches at {failed[:10]}")
    return out, failed

## 6 · Experiment A — reliability

Two independent passes over the same 200 items. Primary metric is **macro
per-category Cohen's κ**, comparable to the single-label κ = 0.560 baseline.
Jaccard is reported alongside but is not chance-corrected, so it is not the gate.

~40 calls.

In [ ]:
print("Pass 1"); m1, f1 = run_pass(pilot, tag="[A1]")
print("Pass 2"); m2, f2 = run_pass(pilot, tag="[A2]")

S1 = [as_set(m1.get(str(i + 1))) for i in range(len(pilot))]
S2 = [as_set(m2.get(str(i + 1))) for i in range(len(pilot))]
both = [(a, b) for a, b in zip(S1, S2) if a and b]

mean_jacc = float(np.mean([len(a & b) / len(a | b) for a, b in both]))
exact     = float(np.mean([a == b for a, b in both]))
mean_size = float(np.mean([len(a) for a in S1 if a]))

# Macro per-category Cohen's kappa
kappas = {}
for cat in category_names:
    a = np.array([cat in s for s in S1]); b = np.array([cat in s for s in S2])
    po = (a == b).mean()
    pe = a.mean() * b.mean() + (1 - a.mean()) * (1 - b.mean())
    kappas[cat] = (po - pe) / (1 - pe) if pe < 1 else np.nan

macro_kappa = float(np.nanmean(list(kappas.values())))
SINGLE_LABEL_KAPPA = 0.560

print(f"\n{'=' * 68}")
print("EXPERIMENT A — RELIABILITY")
print("=" * 68)
print(f"  items compared            : {len(both)}")
print(f"  MACRO PER-CATEGORY KAPPA  : {macro_kappa:.3f}   <-- primary metric")
print(f"  single-label kappa        : {SINGLE_LABEL_KAPPA:.3f}   (notebook 03)")
print(f"  mean Jaccard              : {mean_jacc:.3f}   (not chance-corrected)")
print(f"  exact set match           : {exact:.1%}")
print(f"  mean label-set size       : {mean_size:.2f}")
print(f"  labels dropped by filter  : {count_dropped(m1, len(pilot))}")

print("\n  set size distribution:")
tot = len([a for a in S1 if a])
for k, v in sorted(Counter(len(a) for a in S1 if a).items()):
    print(f"    {k}: {v:>4}  ({v/tot:>5.1%})  {'█' * int(30*v/tot)}")

print("\n  per-category kappa:")
for cat, k in sorted(kappas.items(), key=lambda kv: -(kv[1] if kv[1] == kv[1] else -9)):
    n1 = sum(cat in s for s in S1)
    print(f"    {cat[:34]:<36} {k:>6.3f}   (n={n1})")

In [ ]:
c1_pass = macro_kappa > SINGLE_LABEL_KAPPA
c2_pass = 1.0 <= mean_size <= 3.0

print("=" * 68)
print("CONDITIONS 1 AND 2")
print("=" * 68)
print(f"  1. macro kappa > {SINGLE_LABEL_KAPPA:.3f}   : {macro_kappa:.3f}   {'PASS' if c1_pass else 'FAIL'}")
print(f"  2. set size in [1.0, 3.0] : {mean_size:.2f}    {'PASS' if c2_pass else 'FAIL'}")
print()
if c1_pass and c2_pass:
    print("  -> Proceed to Section 8 (discrimination).")
elif mean_size > 3.0:
    print("  -> Sets inflated; agreement is degenerate. Tighten the prompt first.")
else:
    print("  -> Multi-label does not beat single-label. Fall back to the")
    print("     6-category merge (kappa = 0.625) from notebook 03.")

## 7 · Per-category probe *(optional — already answered)*

Asks the model eight independent yes/no questions per item instead of requesting
a list, removing any list-length prior from the format entirely.

**Result from 2026-07-31:** mean 1.50 categories marked true, versus 1.58 under
the list format, with 4% of items at three categories. The 1–2 concentration is
the model's honest judgment — several categories are structurally inapplicable to
any given problem (Representation Misreading needs a graph, Appearance-Based
Spatial needs a figure).

Set `RUN_PROBE = True` in Section 1 to reproduce.

In [ ]:
PROBE_TEMPLATE = """For each numbered item, judge EVERY category independently.

For each of the 8 categories, answer whether that category applies to this \
specific wrong answer. These are independent yes/no judgments — a category \
applying does not make another less likely, and there is no expected number of \
"yes" answers.

CATEGORIES:
{cats}

Return ONLY JSON, one object per item, all 8 categories as keys with true/false.
Use the exact category names given above as keys.
{{"1": {{"<category name>": true, "<category name>": false}}, "2": {{}}}}

ITEMS:
{items}"""

if RUN_PROBE:
    probe = pilot.head(50)
    out = {}
    for start in range(0, len(probe), BATCH_SIZE):
        try:
            out.update(label_batch(probe.iloc[start:start+BATCH_SIZE], start, PROBE_TEMPLATE))
        except Exception as e:
            print(f"\n  probe batch {start} failed: {e}")
        print(f"\r  probed {len(out)}/{len(probe)}", end="")
    print()

    counts = []
    for i in range(len(probe)):
        d = out.get(str(i + 1)) or {}
        counts.append(sum(1 for k, v in d.items() if v is True and k in category_names))

    print(f"\n  mean categories marked true : {np.mean(counts):.2f}")
    print(f"  list-format mean set size   : {mean_size:.2f}")
    for k, v in sorted(Counter(counts).items()):
        print(f"    {k}: {v:>3}  ({v/len(counts):>5.1%})")
else:
    print("Skipped (RUN_PROBE = False). Established result: mean 1.50 vs 1.58 list format.")

## 8 · Experiment B — do profiles discriminate?

**This is the open question.**

Reliable labels are worthless if every student ends up with the same profile. The
pilot labels *(problem, wrong answer)* pairs rather than students, so this needs
its own sample: pick students, then label the pairs those students produced.

The next cell prints the pair count and call estimate **before** spending
anything. Lower `N_STUDENTS` in Section 1 if it looks steep.

In [ ]:
all_wrong = pd.concat([
    fillin_wrong[['user_id', 'problem_id', 'answer_key']].assign(format='fill-in'),
    mc1_wrong[['user_id', 'problem_id', 'answer_key']].assign(format='mc'),
])

labelable = set(zip(fillin_pairs['problem_id'], fillin_pairs['answer_text'])) | \
            set(zip(mc_pairs['problem_id'], mc_pairs['answer_text']))

all_wrong['pair'] = list(zip(all_wrong['problem_id'], all_wrong['answer_key']))
all_wrong = all_wrong[all_wrong['pair'].isin(labelable)]

counts_per_student = all_wrong.groupby('user_id').size()
eligible = counts_per_student[counts_per_student >= MIN_WRONG].index.tolist()
random.shuffle(eligible)
sample_students = eligible[:N_STUDENTS]

student_rows = all_wrong[all_wrong['user_id'].isin(sample_students)]
needed_pairs = student_rows['pair'].drop_duplicates()

print(f"Eligible students (>= {MIN_WRONG} labelable wrong answers) : {len(eligible):,}")
print(f"Sampled students                                    : {len(sample_students)}")
print(f"Their wrong answers                                 : {len(student_rows):,}")
print(f"UNIQUE pairs needing labels                         : {len(needed_pairs):,}")
print(f"Estimated API calls (batch {BATCH_SIZE})                     : {int(np.ceil(len(needed_pairs)/BATCH_SIZE)):,}")
print(f"\nLabelable wrong answers per student:")
print(student_rows.groupby('user_id').size().describe().round(1).to_string())

### 8.1 · Resolve pairs to triples

In [ ]:
all_triples = pd.concat([fillin_triples, mc_triples], ignore_index=True)
all_triples['pair'] = list(zip(all_triples['problem_id'], all_triples['wrong']))

to_label = (all_triples
            .merge(pd.DataFrame({'pair': needed_pairs.tolist()}), on='pair', how='inner')
            .drop_duplicates('pair')
            .reset_index(drop=True))

resolved = len(to_label) / len(needed_pairs)
print(f"Triples resolved : {len(to_label):,} of {len(needed_pairs):,}  ({resolved:.1%})")
if resolved < 0.8:
    print("WARNING: under 80% resolved — check the pair key join before continuing.")

### 8.2 · Label them

In [ ]:
print(f"Labeling {len(to_label):,} pairs (single pass)")
mb, failed_b = run_pass(to_label, tag="[B]")

to_label['labels'] = [list(as_set(mb.get(str(i + 1)))) for i in range(len(to_label))]
pair_labels = {p: l for p, l in zip(to_label['pair'], to_label['labels']) if l}

covered = student_rows['pair'].isin(pair_labels).mean()
print(f"\nPairs labeled       : {len(pair_labels):,}")
print(f"Student-row coverage: {covered:.1%}")
if covered < 0.8:
    print("WARNING: low coverage will bias profiles toward whichever pairs labeled cleanly.")

### 8.3 · Build per-student profiles

In [ ]:
profiles = {}
for sid, grp in student_rows.groupby('user_id'):
    cnt, n_obs = Counter(), 0
    for pair in grp['pair']:
        labs = pair_labels.get(pair)
        if labs:
            n_obs += 1
            for l in labs:
                if l in category_names:
                    cnt[l] += 1
    if n_obs >= 10:
        profiles[sid] = {'n_obs': n_obs,
                         'share': {c: cnt.get(c, 0) / n_obs for c in category_names}}

P = pd.DataFrame({sid: p['share'] for sid, p in profiles.items()}).T[category_names]
n_obs = pd.Series({sid: p['n_obs'] for sid, p in profiles.items()})

print(f"Profiles built for {len(P)} students")
print(f"Observations per student: median {n_obs.median():.0f}, range {n_obs.min()}–{n_obs.max()}")
print("\nPopulation mean share:")
for cat in category_names:
    print(f"  {cat[:34]:<36} {P[cat].mean():>6.1%}   (sd {P[cat].std():.3f})")

### 8.4 · The discrimination test

For each category, the **overdispersion ratio** — observed variance of that
category's share across students, divided by the variance expected if every
student drew from a single population rate.

- **≈ 1** — students indistinguishable on this category; no individual signal
- **> 1.2** — students genuinely differ
- **> 2** — strong individual signal

In [ ]:
print("OVERDISPERSION BY CATEGORY")
print("=" * 76)
print(f"{'Category':<36} {'mean':>7} {'obs var':>9} {'exp var':>9} {'ratio':>7}")
print("-" * 76)

od = {}
mean_n = n_obs.mean()
for cat in category_names:
    p = P[cat].mean()
    obs_var = P[cat].var()
    exp_var = p * (1 - p) / mean_n if 0 < p < 1 else np.nan
    ratio = obs_var / exp_var if exp_var and exp_var == exp_var else np.nan
    od[cat] = ratio
    flag = '' if ratio != ratio else ('  STRONG' if ratio > 2 else ('  ok' if ratio > 1.2 else '  FLAT'))
    print(f"{cat[:34]:<36} {p:>6.1%} {obs_var:>9.5f} {exp_var:>9.5f} {ratio:>7.2f}{flag}")

overall = float(np.nanmean(list(od.values())))
c3_pass = overall > 1.0

print("-" * 76)
print(f"{'MEAN OVERDISPERSION':<36} {'':>7} {'':>9} {'':>9} {overall:>7.2f}")
print("=" * 76)
print(f"  3. Profiles discriminate : {overall:.2f}   {'PASS' if c3_pass else 'FAIL'}")
print("=" * 76)
if c3_pass:
    print("  Students differ more than chance. Profiles carry individual signal —")
    print("  which is what the LSTM would learn to predict.")
else:
    print("  Profiles collapse toward the population average. Labels are reliable")
    print("  but uninformative: every student looks the same.")

### 8.5 · Split-half profile stability

The overdispersion statistic in 8.4 assumes independent draws at a common sample
size. Neither holds exactly: students range from 32 to 132 observations, and a
student's wrong answers cluster by knowledge component rather than arriving
independently. Both inflate the ratio, so 2.30 is probably optimistic.

This test makes no distributional assumptions. Each student's labeled answers are
split in half at random, a profile is built from each half, and the question is
whether a student's two halves resemble **each other** more than they resemble
**other students'** halves.

**Profiles are mean-centred first.** Without centring every student looks alike,
because they all share the population-average shape. Centring isolates the part
that is individual.

Three outputs:

- **Identification rate** — match each student's first-half profile to the
  nearest second-half profile among all students. Chance is 1/N.
- **Within vs between similarity** — the gap is the individual signal.
- **Per-category split-half correlation**, Spearman-Brown corrected, directly
  comparable to the overdispersion column.

No API calls; re-partitions labels already in memory.

In [ ]:
# ---- Section 8.5 · Split-half profile stability -----------------------------
N_SPLITS = 20      # repeat the random split; averages out split luck
MIN_HALF = 10      # minimum observations per half

student_obs = {}
for sid, grp in student_rows.groupby('user_id'):
    obs = [pair_labels[p] for p in grp['pair'] if p in pair_labels and pair_labels[p]]
    if len(obs) >= 2 * MIN_HALF:
        student_obs[sid] = obs

sids = sorted(student_obs)
N = len(sids)
print(f"Students with >= {2*MIN_HALF} labeled answers: {N}")
print(f"Observations per student: median "
      f"{int(np.median([len(v) for v in student_obs.values()]))}, "
      f"range {min(len(v) for v in student_obs.values())}-"
      f"{max(len(v) for v in student_obs.values())}")


def profile_from(obs_list):
    cnt = Counter()
    for labs in obs_list:
        for l in labs:
            if l in category_names:
                cnt[l] += 1
    n = max(len(obs_list), 1)
    return np.array([cnt.get(c, 0) / n for c in category_names], dtype=float)


def cos_sim(A, B):
    An = A / (np.linalg.norm(A, axis=1, keepdims=True) + 1e-12)
    Bn = B / (np.linalg.norm(B, axis=1, keepdims=True) + 1e-12)
    return An @ Bn.T


ident_rates, within_means, between_means = [], [], []
cat_corrs = {c: [] for c in category_names}
rng = np.random.default_rng(RANDOM_SEED)

for _ in range(N_SPLITS):
    A = np.zeros((N, len(category_names)))
    B = np.zeros((N, len(category_names)))
    for i, sid in enumerate(sids):
        obs = list(student_obs[sid])
        rng.shuffle(obs)
        mid = len(obs) // 2
        A[i] = profile_from(obs[:mid])
        B[i] = profile_from(obs[mid:])

    # per-category split-half correlation (uncentred — absolute shares)
    for j, c in enumerate(category_names):
        if A[:, j].std() > 1e-9 and B[:, j].std() > 1e-9:
            cat_corrs[c].append(np.corrcoef(A[:, j], B[:, j])[0, 1])

    # centre on the population mean of this split, then compare
    Ac, Bc = A - A.mean(0), B - B.mean(0)
    S = cos_sim(Ac, Bc)

    ident_rates.append(float((S.argmax(axis=1) == np.arange(N)).mean()))
    within_means.append(float(np.diag(S).mean()))
    off = S[~np.eye(N, dtype=bool)]
    between_means.append(float(off.mean()))

ident   = float(np.mean(ident_rates))
within  = float(np.mean(within_means))
between = float(np.mean(between_means))
chance  = 1.0 / N

print(f"\n{'=' * 76}")
print(f"SPLIT-HALF STABILITY   ({N_SPLITS} random splits, {N} students)")
print("=" * 76)
print(f"  Identification rate      : {ident:>6.1%}   (chance {chance:.1%}, "
      f"{ident/chance:.1f}x)")
print(f"  Within-student cosine    : {within:>6.3f}")
print(f"  Between-student cosine   : {between:>6.3f}")
print(f"  Separation               : {within - between:>6.3f}")

print(f"\n{'=' * 76}")
print("PER-CATEGORY SPLIT-HALF CORRELATION vs OVERDISPERSION")
print("=" * 76)
print(f"{'Category':<36} {'r':>7} {'r_SB':>7} {'overdisp':>9}")
print("-" * 76)
sb_all = {}
for c in category_names:
    r = float(np.mean(cat_corrs[c])) if cat_corrs[c] else np.nan
    r_sb = 2 * r / (1 + r) if r == r and r > -1 else np.nan   # Spearman-Brown
    sb_all[c] = r_sb
    flag = ''
    if r_sb == r_sb:
        flag = '  STABLE' if r_sb > 0.6 else ('  weak' if r_sb > 0.3 else '  NOISE')
    print(f"{c[:34]:<36} {r:>7.3f} {r_sb:>7.3f} {od.get(c, float('nan')):>9.2f}{flag}")

mean_sb = float(np.nanmean(list(sb_all.values())))
print("-" * 76)
print(f"{'MEAN':<36} {'':>7} {mean_sb:>7.3f}")

c3b_pass = ident > 5 * chance and (within - between) > 0.15
print("=" * 76)
print(f"  Split-half check : {'PASS' if c3b_pass else 'REVIEW'}")
print("=" * 76)
if c3b_pass:
    print("  A student's two halves match each other far above chance. Profiles")
    print("  are stable individual signal, not an artifact of the binomial")
    print("  assumption in 8.4.")
else:
    print("  Halves do not match reliably. The overdispersion in 8.4 was likely")
    print("  inflated by within-student clustering — treat condition 3 as unproven.")

## 9 · TF-IDF weighting

Raw shares over-represent categories that fire on most wrong answers. Weighting
by inverse population frequency surfaces what is *distinctive* about a student
rather than what is common to everyone.

Kept as a display-time transform rather than folded into the training target, so
it stays tunable without retraining.

In [ ]:
idf = {c: np.log(1 / max(P[c].mean(), 1e-6)) for c in category_names}
W = P.mul(pd.Series(idf), axis=1)
W = W.div(W.sum(axis=1).replace(0, np.nan), axis=0)

print("IDF weights (higher = rarer = more diagnostic)")
for c, v in sorted(idf.items(), key=lambda kv: -kv[1]):
    print(f"  {c[:34]:<36} {v:>5.2f}")

print("\n\nEXAMPLE PROFILES — raw vs weighted")
print("=" * 76)
for sid in list(P.index)[:3]:
    print(f"\nStudent {sid}  ({profiles[sid]['n_obs']} labeled wrong answers)")
    raw, wt = P.loc[sid].sort_values(ascending=False), W.loc[sid].sort_values(ascending=False)
    print(f"  {'by raw share':<40} {'by weighted share'}")
    for i in range(3):
        print(f"    {raw.index[i][:26]:<28} {raw.iloc[i]:>5.1%}"
              f"      {wt.index[i][:26]:<28} {wt.iloc[i]:>5.1%}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))

sizes = Counter(len(a) for a in S1 if a)
axes[0].bar(list(sizes.keys()), list(sizes.values()), color='#2c5f8a')
axes[0].set_xlabel('labels per item'); axes[0].set_ylabel('items')
axes[0].set_title(f'Label-set size (mean {mean_size:.2f})', fontsize=11)

ods = pd.Series(od).dropna().sort_values()
axes[1].barh([c[:24] for c in ods.index], ods.values, color='#6b9dbf')
axes[1].axvline(1, color='crimson', ls='--', lw=1, label='no discrimination')
axes[1].set_xlabel('overdispersion ratio')
axes[1].set_title('Do students differ per category?', fontsize=11)
axes[1].tick_params(labelsize=7); axes[1].legend(fontsize=7)

axes[2].boxplot([P[c].values for c in category_names],
                labels=[c[:12] for c in category_names])
axes[2].set_ylabel('share of wrong answers')
axes[2].set_title('Profile spread across students', fontsize=11)
axes[2].tick_params(axis='x', rotation=90, labelsize=7)

plt.tight_layout()
plt.savefig('stage1b_multilabel_validation.png', dpi=200, bbox_inches='tight')
plt.show()

## 10 · Verdict

In [ ]:
verdict = {
    'generated_utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    'model': LABELING_MODEL,
    'n_pilot': len(both),
    'macro_kappa': macro_kappa,
    'single_label_kappa_baseline': SINGLE_LABEL_KAPPA,
    'mean_jaccard': mean_jacc,
    'exact_set_match': exact,
    'mean_set_size': mean_size,
    'per_category_kappa': {k: (None if v != v else float(v)) for k, v in kappas.items()},
    'n_students_profiled': int(len(P)),
    'student_row_coverage': float(covered),
    'mean_overdispersion': overall,
    'overdispersion_by_category': {k: (None if v != v else float(v)) for k, v in od.items()},
    'conditions': {'reliability': bool(c1_pass),
                   'set_size': bool(c2_pass),
                   'discrimination': bool(c3_pass)},
}

with open('stage1b_multilabel_verdict.json', 'w') as f:
    json.dump(verdict, f, indent=2)

allpass = c1_pass and c2_pass and c3_pass
print("=" * 76)
print("VERDICT")
print("=" * 76)
print(f"  1. Reliability   macro kappa {macro_kappa:.3f} vs {SINGLE_LABEL_KAPPA:.3f}   {'PASS' if c1_pass else 'FAIL'}")
print(f"  2. Set size      {mean_size:.2f} in [1.0, 3.0]              {'PASS' if c2_pass else 'FAIL'}")
print(f"  3. Discrimination overdispersion {overall:.2f} > 1.0      {'PASS' if c3_pass else 'FAIL'}")
print("-" * 76)
if allpass:
    print("  ALL PASS -> adopt multi-label.")
    print("  Next: full run at ~20,602 pairs, then LSTM with 8 sigmoid outputs,")
    print("  BCE loss, pos_weight from the category frequencies measured here.")
    print("  Unlabeled rows (select-all, fill-in tail) masked in the loss.")
else:
    print("  NOT ALL PASS. Failing:")
    if not c1_pass: print("    - reliability: consider the 6-category merge instead")
    if not c2_pass: print("    - set size: prompt is hedging; tighten and re-run Section 6")
    if not c3_pass: print("    - discrimination: profiles carry no individual signal.")
    print("      A substantive negative result — record it either way.")
print("=" * 76)
print("Saved stage1b_multilabel_verdict.json + stage1b_multilabel_validation.png")

if IN_COLAB:
    from google.colab import files
    files.download('stage1b_multilabel_verdict.json')

---

## Next steps

Record the outcome in `capstone/stage1-pilot-findings.md` regardless of result.

**All pass** — scale to the full ~20,602 pairs (notebook 05), then build the LSTM.

**Condition 3 fails** — misconception profile is not a property that varies
between students at this granularity. That is a real negative result and belongs
in the report; fall back to the 6-category single-label optimum.

---

*Notebook 02 produced the taxonomy · 03 measured single-label reliability ·
04 tests the multi-label alternative.*